In [1]:

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import pylab
import matplotlib as mpl
import pandas as pd
import math


class QGGraphError(Exception):
    #涵盖：输入电流的两节点间不边的异常
    pass


class QGNode():
    def __init__(self, name, decay=0.8, r=0) -> None:

        super().__init__()
        self.name = name
        #self.father=None
        #self.attr= production/company
        self.decay = decay  #衰减系数
        self.r = r  #r，节点净库存流
        self.nbr = {}  #neighbor+关系 par/raw/st
        self.adj = []
        self.par = []
        self.pred = []
        self.parent=[]
        self.children=[]

    def __str__(self):
        return self.name
    def __repr__(self):
        return self.name
    def update_r(self, change):
        self.r = round(self.r + change, 2)

    def get_item(self):
        return self.name
    def add_parent(self,name):
        self.parent
    def add_adj(self, name, re, i=0):
        self.nbr[name] = {'re': re, 'i': i}
        self.adj.append(name)

    def add_parity(self, name, re, i=0):
        self.nbr[name] = {'re': re, 'i': i}
        self.par.append(name)

    def add_pred(self, name, re, i=0):
        self.nbr[name] = {'re': re, 'i': i}
        self.pred.append(name)

    def __dict__(self):
        return {self.name: {x for x in self.nbr}}

    def get_r(self):
        return self.r

    def get_edge_list(self):
        # ('B', 'A', {'re': 'raw','i': 0}),
        edge_list = []
        for neighbor in self.nbr:
            edge_list.append((self.name, neighbor, self.nbr[neighbor]))
        return edge_list


class QGGraph():
    def __init__(self, ):
        self.node_dict = {}
        self.edge_dict = {}
        self.updated_node_list = []
        self.updated_edge_list = []
        self.num_nodes = 0
        self.num_edges = 0
        self.c_par = -0.5
        self.c_node = 0.8
        self.source_node=''
        self.source_input=0
    #构建和检索部分
    def add_Node(self, name):
        new_Node = QGNode(name)
        self.node_dict[name] = new_Node
        self.num_nodes += 1
        return new_Node

    def get_Node(self, name) -> dict:
        if name in self.node_dict:
            return self.node_dict[name]
        else:
            return None

    def get_all_Nodes(self) -> dict:
        dic = {}
        for node in self.node_dict:
            dic[node] = self.node_dict[node].get_r()
        return dic

    def add_Edge(self, f, t, re, i):
        if f not in self.node_dict:
            nn = self.add_Node(f)
        if t not in self.node_dict:
            nn = self.add_Node(t)
        if re in ('raw', 'st'):
            self.node_dict[f].add_adj(t, re, i)
            self.node_dict[t].add_pred(f, re, i)
        if re == 'par':
            self.node_dict[f].add_parity(t, re, i)
            self.node_dict[t].add_parity(f, re, i)
        self.edge_dict[(f, t)] = {
            're': re,
            'i': i
        }  #在图里保存并更新所有边的信息,点只存变量r，不存边的i
        self.num_edges += 1

    def get_all_Edges(self):
        return self.edge_dict

    # networkx 可视化部分
    def build_nx_dic(self):
        nx_node_list = []
        for n in self.node_dict:
            dic = {'node': self.node_dict[n], 'r': self.node_dict[n].get_r()}
            nx_node_list.append((n, dic))
        nx_edge_list = []
        for e in self.edge_dict:
            f, t = e
            nx_edge_list.append((f, t, self.edge_dict[e]))
        return nx_node_list, nx_edge_list

    def build_networkx(self):
        node, edge = self.build_nx_dic()
        G = nx.DiGraph()
        G.add_nodes_from(node)
        G.add_edges_from(edge)
        return G

    def draw_networkx(self):
        G = self.build_networkx()

        options = {
            "font_size": 15,
            "node_size": 1100,
            "node_color": "w",
            "edgecolors": "black",
            "linewidths": 0.5,
            "width": 1,
        }
        pos = nx.spring_layout(G, k=10000,
                               seed=6)  # Seed layout for reproducibility
        nx.draw(
            G,
            pos,
            **options,
        )
        node_labels = {}
        for node in G.nodes:
            node_labels[node] = str(node) + ': ' + str(G.nodes[node]['r'])
        nx.draw_networkx_labels(G, pos, labels=node_labels)
        edge_labels = {}
        for edge in G.edges:
            edge_labels[edge] = str(G[edge[0]][edge[1]]['re']) + ' ' + str(
                G[edge[0]][edge[1]]['i'])
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
        plt.show()

    # 计算、更新部分
    #轮子
    def update_Edge(self, f, t, i):
        if (f, t) in self.edge_dict:
            assert (f, t) not in self.updated_edge_list, ('该边已被更新过:', f, t)
            self.edge_dict[f, t]['i'] = round(i + self.edge_dict[f, t]['i'], 2)
            self.updated_edge_list.append((f, t))
        elif (t, f) in self.edge_dict:
            assert (t, f) not in self.updated_edge_list, ('该边已被更新过:', t, f)
            self.edge_dict[t, f]['i'] = round(i + self.edge_dict[t, f]['i'], 2)
            self.updated_edge_list.append((t, f))
        else:
            raise QGGraphError('更新的边不在图中存在:', f, t)

    def update_Node(self, name, r):
        assert name in self.node_dict, ('该节点不在图中:', name)
        assert name not in self.updated_node_list, ('该节点已被更新:', name)
        self.node_dict[name].update_r(r)
        self.updated_node_list.append(name)

    def get_pred_par(self, name):

        #假定互为同位的节点被全部输入，即两两间都为同位关系,获得分组列表
        query = []
        result = []
        for pred in self.node_dict[name].pred:
            group = [pred]
            if pred not in query:
                for par in self.node_dict[pred].par:
                    if par not in query:
                        if name in self.node_dict[par].adj:
                            group.append(par)
                            query.append(par)
                result.append(group)
                query.append(pred)
        return result

    def get_adj_par(self, name):
        query = []
        result = []
        for adj in self.node_dict[name].adj:
            group = [adj]
            if adj not in query:
                for par in self.node_dict[adj].par:
                    if par not in query:
                        if name in self.node_dict[par].adj:
                            group.append(par)
                            query.append(par)
                result.append(group)
                query.append(adj)
        return result

    #输入和响应
    def update_graph(self, input):
        self.add_source(input)
        self.source_node=input['f']
        self.source_input=input['i']
        self.update(f=input['f'], t=input['t'], i_in=input['i'])
        print('本次输入更新的边为:', self.updated_edge_list)
        print('本次输入更新的点为:', self.updated_node_list)
        self.updated_node_list = []
        self.updated_edge_list = []
    def add_source(self, input):
        #input={'f':'B','t':'A','i':20}
        f = input['f']
        t = input['t']
        i = input['i']
        self.update_Edge(f=f, t=t, i=i)
    '''def bfs(f,t,i_in):
        t1,i_out=update(f,t,i_in)'''

    def update(self, f, t, i_in) -> None:
        '''
        先分组，再计算，再递归传导
        对一个点产生影响的只有一个源（或者经过同位衰减的一组源）
        f='b',t='a',i_in=20
        此处f，t为影响力传播，非边的方向,i为ft之间的影响力，同时为t的i_in更新前的初值
        只对t更新，然后由t传播(与边的方向无关)
        '''
        if t==self.source_node:
            i_in=self.source_input
        


        c_node = self.c_node
        c_par = self.c_par
        preds = self.get_pred_par(t)
        adjs = self.get_adj_par(t)
        f_node = self.node_dict[f]
        t_node = self.node_dict[t]
        if f in t_node.pred:  
            #f为t的上游时，f-t是源，找上游组
            for group in preds:
                if f in group:
                    if len(group)>1:
                        group.remove(f)
                        for other in group:
                            if (other,t) not in self.updated_edge_list:
                                self.update_Edge(other, t, round(c_par * i_in / len(group),2))
                        i_in = i_in * (1 + c_par)
        else:
            #否则相反，找下游组
            for group in adjs:
                if f in group:
                    if len(group)>1:
                        group.remove(f)
                        for other in group:
                            if (t,other) not in self.updated_edge_list:
                                self.update_Edge(t, other, round(c_par * i_in / len(group),2))
                        i_in = i_in * (1 + c_par)
        #更新ir
        if t not in self.updated_node_list:
            self.update_Node(t,r=round(i_in*(1-c_node),2))
        else:
            pass
        #对其他所有组更新iout
        i_out=round(i_in*c_node,2)
        
        for group in preds:
            num=len(group)
            for other in group:
                if (other,t) not in self.updated_edge_list:
                    self.update_Edge(other,t,round(i_out/num,2))
        for group in adjs:
            num=len(group)
            for other in group:
                if(t,other) not in self.updated_edge_list:
                    self.update_Edge(t,other,round(i_out/num,2))
        #更新相邻节点 #BFS
        return (t,i_out)
        '''others=self.node_dict[t].adj+self.node_dict[t].pred

        for pred in self.node_dict[t].pred:
            if pred not in self.updated_node_list:
                self.recursive_update(t,pred,i_out)
        for adj in self.node_dict[t].adj:
            if adj not in self.updated_node_list:
                self.recursive_update(t,adj,i_out)
        
        #更新i_in,如果有上游同位则同位衰减
        for par in self.node_dict[start].par:
            for adj in self.node_dict[par].adj:
                if adj in g.node_dict[start].adj:
                    if (par, adj) not in self.updated_edge_list:
                        #print(par)
                        self.update_Edge(par, adj, i=i_out * c_par)
            for pred in self.node_dict[par].pred:
                if pred in g.node_dict['B'].pred:
                    if(pred,par) not in self.updated_edge_list:
                        self.update_Edge(pred,par,i=i_in*c_par)
        i_out = i_in * c_node
        #更新其他边，如果有同位则分组
        '''



    '''def recursive_update(self, start):
        #start='B' start=f
        c_node = self.c_node
        c_par = self.c_par
        i_in=0
        #print('更新节点',start)
        #更新节点流经影响力
        for edge in self.edge_dict:
            if start in edge:
                i_in+=self.edge_dict[edge]['i']
        #print(start,i_in)
        i_out=round(i_in*c_node,2)
        #print(start,i_out)
        #更新节点所受影响力
        if start not in self.updated_node_list:
            if i_in !=0:  #此处写法存疑，会不会无限递归，是否有需要（是否真的存在例子中先F后K导致无强度）
                self.update_Node(start, i_in * (1 - c_node))
            else:
                return None
        #更新节点其他各边，出现同位时平分影响力
        for pred in self.node_dict[start].pred:
            divided_par=[pred]
            for par in self.node_dict[pred].par:
                if par in self.node_dict[start].pred:
                    divided_par.append(par)
            #divided_par 中存储需要平分影响力的边，如果没有同位则长度为1
            for pred in divided_par:
                #print(pred)
                if (pred,start) not in self.updated_edge_list:
                    self.update_Edge(pred,start,i=round(i_out/len(divided_par),2))
        for adj in self.node_dict[start].adj:
            divided_par=[adj]
            for par in self.node_dict[adj].par:
                if par in self.node_dict[start].adj:
                    divided_par.append(par)
            #divided_par 中存储需要平分影响力的边，如果没有同位则长度为1
            for adj in divided_par:
                #print(adj)
                if (start,adj) not in self.updated_edge_list:
                    self.update_Edge(start,adj,i=round(i_out/len(divided_par),2))
        #更新同位，且有共同pre/adj的边
        for par in self.node_dict[start].par:
            for adj in self.node_dict[par].adj:
                if adj in g.node_dict[start].adj:
                    if (par,adj) not in self.updated_edge_list:
                        #print(par)
                        self.update_Edge(par,adj,i=i_out*c_par)
            for pred in self.node_dict[par].pred:
                if pred in g.node_dict['B'].pred:
                    if(pred,par) not in self.updated_edge_list:
                        self.update_Edge(pred,par,i=i_in*c_par)
        #先更新抑制型同位
        for nbr in self.node_dict[start].par:
            if nbr not in self.updated_node_list:
                self.recursive_update(start=nbr)
        for nbr in self.node_dict[start].nbr:
            if nbr not in self.updated_node_list:
                self.recursive_update(start=nbr)'''


In [2]:
n=QGNode('n')
print(n)

n


In [3]:
G.adj

NameError: name 'G' is not defined

In [4]:
G=nx.DiGraph()
G.add_node(n)
G.add_node(3)
G.add_edge(n,3,rela='compose')
G

In [5]:
G[n]

AtlasView({3: {'rela': 'compose'}})

In [6]:
G.nodes(True)

NodeDataView({n: {}, 3: {}})

In [7]:
G.edges(data=True)

OutEdgeDataView([(n, 3, {'rela': 'compose'})])

In [8]:
print(n)

n


In [9]:
for node in G.pred.items():
    print(node)

(n, AtlasView({}))
(3, AtlasView({n: {'rela': 'compose'}}))


In [10]:
G[n][3]

{'rela': 'compose'}

In [11]:
report=eval('n')
report

n

In [12]:
report.__class__

__main__.QGNode

In [13]:
G.adj

AdjacencyView({n: {3: {'rela': 'compose'}}, 3: {}})

In [14]:
subg=nx.subgraph_view(G,filter_node=nx.filters.hide_nodes([n,3]))
subg.edges()


OutEdgeView([])

In [15]:
g1=nx.reverse_view(G)
g1.adj

AdjacencyView({n: {}, 3: {n: {'rela': 'compose'}}})

In [16]:
matrix=nx.adjacency_matrix(G)

In [17]:
nx.ancestors(G,3)

{n}

In [18]:
nx.is_directed_acyclic_graph(G)

True

In [35]:
from functools import wraps

def timer(a_func):
    @wraps(a_func)
    def wrap(*args, **kwargs):
        import time
        start=time.time()
        r=a_func(*args, **kwargs)
        end=time.time()
        t=end-start
        print('consumed time: %.3fs'%t)
        return r
    return wrap


consumed time: 0.000s


2